# PHYS 3810: setup preflight

Run **Kernel → Restart Kernel and Run All Cells**. This small local check needs no IBM account, token, GPU, or internet after installation. A red traceback means the check failed; keep its text when asking for help.

Select the Python interpreter inside this repository's `.venv`. See [the setup guide](../SETUP.md) for installation and kernel selection. These are environment checks, not the five algorithm lessons.


In [ ]:
import sys
from importlib.metadata import version

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12), "Select CPython 3.12."
for package, expected in {"qiskit": "2.6.1", "qiskit-aer": "0.17.2"}.items():
    actual = version(package)
    print(f"{package}: {actual}")
    assert actual == expected, f"Expected {package}=={expected}; check the selected kernel."
print("PASSED: Python and Qiskit versions")


## Prepare and check a Bell state

The Hadamard gate puts qubit 0 into a superposition. The controlled-X copies its computational-basis value onto qubit 1, giving $ (|00\rangle + |11\rangle)/\sqrt{2} $. Qiskit writes qubit 0 on the **right** in a two-bit label. The amplitude vector order is `00`, `01`, `10`, `11`.

We first check exact amplitudes, probabilities, and the correlation $\langle ZZ\rangle=1$. Exact simulation is a mathematical reference; measurement counts fluctuate.


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
state = Statevector.from_instruction(bell)
expected_amplitudes = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)
np.testing.assert_allclose(state.data, expected_amplitudes, rtol=0, atol=1e-12)
np.testing.assert_allclose(state.probabilities(), [0.5, 0, 0, 0.5], rtol=0, atol=1e-12)
np.testing.assert_allclose(state.expectation_value(SparsePauliOp("ZZ")), 1.0, rtol=0, atol=1e-12)
print("Bell probabilities:", state.probabilities())
print("PASSED: exact Bell amplitudes, probabilities, and <ZZ> = 1")


In [ ]:
from qiskit.primitives import StatevectorSampler

measured_bell = bell.measure_all(inplace=False)
shots = 1024
result = StatevectorSampler(seed=1234).run([measured_bell], shots=shots).result()[0]
counts = result.data.meas.get_counts()
assert sum(counts.values()) == shots
assert set(counts) == {"00", "11"}
# Six standard deviations for a binomial probability of 1/2.
assert abs(counts["00"] - shots / 2) <= 6 * np.sqrt(shots * 0.25)
print("Counts:", counts)
print("PASSED: 1024 shots, Bell outcomes only, approximately half each")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit.visualization import plot_histogram

# These exercise the drawing dependencies students will use in later labs.
figure = bell.draw("mpl")
display(figure)
plt.close(figure)
figure = plot_histogram(counts, title="Bell measurement counts")
display(figure)
plt.close(figure)
print("PASSED: circuit and histogram rendering")


## Ready for the local lab

Expected: Python `3.12.x`, Qiskit `2.6.1`, Aer `0.17.2`, probabilities `[0.5 0.  0.  0.5]`, 1024 total shots split between `00` and `11`, two figures, and all `PASSED` messages. Exact random counts need not match another computer.

Run `python scripts/validate_classroom.py` from a terminal in the repository folder to also check Aer, offline unit tests, and all currently required classroom notebooks in fresh kernels. It writes executed copies under `artifacts/executed-classroom/`. The optional cloud/GPU examples are skipped; onboarding and Grover are available. QFT, QPE, HHL, and Shor remain pending.
